# Benchmark del Entregable 2

Comparación de los ocho modelos profundos (tres modelos base y cinco del estado del arte) y de los dos
baselines (trivial y regresión logística sobre embeddings) con las funciones de `src/benchmark.py`,
`src/stats.py`, `src/calibration.py`, `src/robustness.py`, `src/interpret.py`, `src/errors.py`,
`src/ceiling.py` y `src/cost.py`. Las preguntas, las pruebas y la regla de decisión son las de
`preregistro_e2.md` y del [capítulo 12](12-preguntas-y-datos.md). Las observaciones pareadas son las
corridas definitivas con la misma semilla y el mismo pliegue de validación; todas las métricas son de
test salvo que se indique. Si faltan resultados, cada sección muestra un aviso y el notebook continúa.

## 1. Configuración

Flags: `RECALCULAR` repite las tablas guardadas en `outputs/benchmark_e2/` (se invalidan solas si cambian
las corridas); `RECALCULAR_GLMM` ajusta el modelo mixto por imagen, que tarda decenas de minutos;
`ESCRIBIR_PLANTILLA_TAXONOMIA` crea `data/taxonomia_errores.csv` si no existe (nunca la sobrescribe).

In [ ]:
RECALCULAR = False
RECALCULAR_GLMM = False
ESCRIBIR_PLANTILLA_TAXONOMIA = True

In [ ]:
import pandas as pd

from src import benchmark, calibration, ceiling, cost, data, errors, evaluation, gradcam, interpret, robustness
from src.report import configure_display, show

configure_display()
cfg = data.load_config()
cfg_b, cfg_st = cfg["benchmark"], cfg["e2_stats"]
grupos = benchmark.e2_groups(cfg)
profundos = grupos["deep_models"]
clases = data.class_names(cfg)
pd.Series({
    "modelos base": ", ".join(grupos["base_models"]),
    "modelos del estado del arte": ", ".join(grupos["sota_models"]),
    "baselines": ", ".join(grupos["baselines"]),
    "métrica primaria": cfg_b["primary_metric"],
    "partición": cfg_b["split"],
    "δ (margen de relevancia)": cfg_b["delta"],
    "δg (margen de sobreajuste)": cfg["e2"]["overfitting_margin"],
    "α": cfg_b["alpha"],
    "hipótesis alternativa": cfg_st["alternative"],
    "bootstrap por lesión": f"{cfg_b['lesion_bootstrap']['n_resamples']} remuestreos, semilla {cfg_b['lesion_bootstrap']['seed']}",
    "ajuste confirmatorio": cfg_b["p_adjust"],
    "ajuste exploratorio": cfg_st["exploratory_p_adjust"],
}, name="valor").to_frame()

### 1.1 Corridas

Corridas definitivas previstas y terminadas por modelo, con su rol, y corridas pareadas con todos los
modelos.

In [ ]:
definitivas = benchmark.load_runs_e2(cfg)
datos_test = benchmark.test_data(definitivas, cfg)
benchmark.availability_e2(definitivas, cfg)

Corridas que cumplen la regla de exclusión del preregistro (pérdida NaN o exactitud balanceada de
validación menor que `e2.divergence.min_val_balanced_accuracy`); se repiten con la siguiente semilla.

In [ ]:
benchmark.divergent_runs(definitivas, cfg)

## 2. Tabla principal

Por modelo, sobre test: cada métrica de `e2_stats.table_metrics` como media ± desviación estándar entre
corridas, el IC 95 % de la métrica primaria por bootstrap sobre lesiones y el costo (parámetros, GFLOPs a
224 × 224 y horas de entrenamiento por corrida).

In [ ]:
costo = cost.cost_table(cost.load_cost(cfg, profundos), cost.cost_from_runs(cfg, profundos))
tabla_principal = benchmark.cached(cfg, definitivas, "tabla_principal",
                                   lambda: benchmark.main_table(definitivas, cfg), RECALCULAR)
benchmark.main_table_formatted(tabla_principal, cfg, costo)

Valores numéricos: media, desviación estándar, IC 95 % por t de Student y, para la métrica primaria, IC
95 % por bootstrap sobre lesiones.

In [ ]:
tabla_principal

Sensibilidad por clase en test (media entre corridas).

In [ ]:
benchmark.per_class_e2(definitivas, cfg, stat="recall")

## 3. Contrastes confirmatorios

Para cada modelo frente a la referencia de su familia, con d = modelo − referencia por corrida pareada
y H1: d > 0: Shapiro-Wilk de d, t pareada con la corrección de Nadeau y Bengio, Wilcoxon exacto,
bootstrap pareado por lesión, TOST con ±δ, Holm dentro de la familia para cada prueba, d de Cohen,
delta de Cliff, A12 de Vargha y Delaney con IC y probabilidad de superioridad. La decisión usa la t de
Nadeau y Bengio si no se rechaza la normalidad y, si se rechaza, exige Wilcoxon y bootstrap
significativos; después, «superior y relevante» si la diferencia es al menos δ, «superior sin
relevancia» si es menor, «equivalente» si el TOST es significativo y «no concluyente» en otro caso.

### 3.1 RQ1: modelos del estado del arte frente al mejor modelo base

In [ ]:
rq1 = benchmark.cached(cfg, definitivas, "rq1", lambda: benchmark.contrast_family(definitivas, cfg, "rq1"), RECALCULAR)
show({"Mejor modelo base": benchmark.best_base(definitivas, cfg), "Resumen": benchmark.contrasts_summary(rq1)})

Todas las columnas del contraste.

In [ ]:
rq1

### 3.2 RQ2: modelos profundos frente a la regresión logística

In [ ]:
rq2 = benchmark.cached(cfg, definitivas, "rq2", lambda: benchmark.contrast_family(definitivas, cfg, "rq2"), RECALCULAR)
benchmark.contrasts_summary(rq2)

In [ ]:
rq2

### 3.3 Pruebas complementarias

McNemar sobre los aciertos por imagen de test en cada corrida pareada, con Holm entre los contrastes de
la familia, y DeLong uno contra el resto por clase, con Holm sobre las siete clases de cada par y
corrida. Resumen entre corridas.

In [ ]:
show({f"McNemar {f.upper()}": benchmark.mcnemar_summary(benchmark.mcnemar_contrasts(definitivas, cfg, f), cfg)
      for f in ("rq1", "rq2")})

In [ ]:
show({f"DeLong {f.upper()}": benchmark.delong_summary(
          benchmark.cached(cfg, definitivas, f"delong_{f}", lambda f=f: benchmark.delong_contrasts(definitivas, cfg, f),
                           RECALCULAR), cfg)
      for f in ("rq1", "rq2")})

## 4. Comparación global

Friedman con la corrección de Iman y Davenport sobre las corridas con todos los modelos
(`e2_stats.global_models`), post-hoc de Nemenyi y de rangos con Holm, y diagrama de diferencia crítica.

In [ ]:
ancho = benchmark.global_wide(definitivas, cfg)
benchmark.global_tests(ancho, cfg)

In [ ]:
benchmark.global_posthoc(ancho, cfg)

In [ ]:
show(benchmark.plot_cd_diagram(ancho, cfg))

Q de Cochran sobre los aciertos por imagen de test de todos los modelos, por corrida pareada.

In [ ]:
cochran = benchmark.cochran_by_run(definitivas, cfg)
show({"Resumen": benchmark.cochran_summary(cochran, cfg), "Por corrida": cochran})

## 5. Bondad de ajuste (tabla A)

Por modelo: binomial exacta de la exactitud frente a la proporción de la clase mayoritaria, AUC-ROC macro
frente a 0,5 y MCC con IC por bootstrap sobre lesiones, pendiente e intercepto de calibración de la clase
predicha, Hosmer-Lemeshow y Z de Spiegelhalter.

In [ ]:
benchmark.cached(cfg, definitivas, 'tabla_a', lambda: benchmark.table_a(definitivas, cfg), RECALCULAR)

## 6. Sobreajuste (tabla B)

Brecha de la métrica primaria entre train sin aumento y validación frente al margen δg (Wilcoxon exacto
unilateral en los dos sentidos), diferencia validación − test con IC por bootstrap sobre lesiones y
coeficiente de variación entre semillas.

In [ ]:
benchmark.cached(cfg, definitivas, 'tabla_b', lambda: benchmark.table_b(definitivas, cfg), RECALCULAR)

## 7. Efectos mixtos y descomposición de la varianza

### 7.1 Por corrida

Modelo lineal mixto de la métrica primaria: efecto fijo del modelo (frente al mejor modelo base) e
intercepto aleatorio de la corrida.

In [ ]:
show(benchmark.mixed_effects_run_level(definitivas, cfg))

### 7.2 Por imagen

Modelo mixto binomial de los aciertos por imagen (Bayes variacional): efecto fijo del modelo y efectos
aleatorios cruzados de la corrida y de la lesión, con la varianza en la escala latente.

In [ ]:
show(benchmark.mixed_effects_image_level(definitivas, cfg, RECALCULAR_GLMM))

### 7.3 Suma de cuadrados

Descomposición de la suma de cuadrados de la métrica primaria en modelo, corrida (semilla, pliegue y su
interacción) y residual.

In [ ]:
benchmark.variance_table(definitivas, cfg)

## 8. Calibración

### 8.1 Temperature scaling

Temperatura ajustada en la validación de cada corrida y métricas de test antes y después: ECE de 15
intervalos, ECE adaptativo, Brier y log-loss (media y desviación estándar entre corridas).

In [ ]:
temperaturas = benchmark.cached(cfg, definitivas, "temperaturas",
                                lambda: calibration.temperature_table(definitivas, cfg, definitivas.models, clases), RECALCULAR)
calibration.temperature_summary(temperaturas)

### 8.2 Diagramas de fiabilidad

Test, todas las corridas juntas, antes (gris) y después (negro) del temperature scaling.

In [ ]:
show(calibration.plot_reliability_before_after(definitivas, cfg, definitivas.models, clases))

## 9. Incertidumbre

### 9.1 Deep ensemble

Media de las probabilidades de todas las corridas de cada modelo frente a la corrida media, con la
entropía predictiva, la información mutua y su AUROC para detectar errores.

In [ ]:
calibration.ensemble_table(definitivas, cfg, profundos, clases)

### 9.2 Predicción conforme

Predicción conforme dividida con la puntuación 1 − p(y | x), calibrada en la validación de cada corrida
(con temperature scaling) y evaluada en test: cobertura frente a 1 − α y tamaño medio del conjunto,
global, por fuente y por clase.

In [ ]:
metadatos = evaluation.load_metadata_or_missing(cfg)
conforme = benchmark.cached(cfg, definitivas, "conforme",
                            lambda: calibration.conformal_results(definitivas, cfg, profundos, clases, metadatos), RECALCULAR)
resumen_conforme = calibration.conformal_summary(conforme, "global")
show(calibration.plot_conformal(resumen_conforme))
resumen_conforme

In [ ]:
show({g: calibration.conformal_summary(conforme, g) for g in ("fuente", "clase")})

## 10. Robustez

Exactitud balanceada en test con ruido gaussiano, desenfoque, compresión JPEG y brillo en tres niveles de
severidad (`robustness`), con los pesos de las corridas de `robustness.runs` (`python -m src.robustness
--all`). Índice de robustez: área bajo la curva normalizada por el valor sin perturbar.

In [ ]:
robustez = robustness.load_results(cfg, profundos)
show(robustness.plot_degradation(robustez))
robustness.robustness_index(robustez)

In [ ]:
robustness.degradation_table(robustez)

## 11. Interpretabilidad

Mapas de saliencia de los `interpretability.top_k` mejores modelos (Grad-CAM para las CNN y attention
rollout para ViT) sobre una muestra común de test (`python -m src.interpret`).

In [ ]:
mejores = interpret.top_models(cfg)
show(mejores)
show({m: gradcam.plot_gradcam_grid(interpret.load_maps(cfg, m), alpha=cfg["evaluation"]["gradcam"]["alpha"])
      for m in mejores})

Prueba de borrado e inserción: AUC de la probabilidad de la clase predicha al borrar o insertar los
píxeles más salientes, frente a un orden aleatorio; y estabilidad de los mapas entre semillas
(correlación de Pearson y Spearman).

In [ ]:
show({"Borrado e inserción": interpret.deletion_insertion_summary(cfg, mejores),
      "Estabilidad entre semillas": interpret.stability_summary(cfg, mejores)})

## 12. Errores

### 12.1 Errores compartidos

Una imagen es un error de un modelo si falla en más de `errors.error_frac` de sus corridas. Imágenes según cuántos
modelos fallan y errores exclusivos y compartidos por modelo.

In [ ]:
fraccion_error = errors.error_fraction(datos_test, profundos)
umbral_error = cfg["errors"]["error_frac"]
show({"Imágenes por número de modelos que fallan": errors.shared_errors_table(fraccion_error, umbral_error),
      "Errores por modelo": errors.errors_per_model(fraccion_error, umbral_error)})

### 12.2 Correlación de errores

Correlación phi entre los indicadores de error por imagen, media entre corridas pareadas.

In [ ]:
correlacion_errores = errors.error_correlation(datos_test, profundos)
show(errors.plot_error_correlation(correlacion_errores))

### 12.3 Ensemble simple

Media de las probabilidades de los `errors.ensemble_top_k` mejores modelos en cada corrida pareada frente
al mejor de ellos.

In [ ]:
ranking = benchmark.ranked_deep_models(definitivas, cfg)
ganancia = errors.ensemble_gain(datos_test, ranking, cfg["errors"]["ensemble_top_k"])
show({"Resumen": errors.ensemble_summary(ganancia), "Por corrida": ganancia})

### 12.4 Subgrupos

Exactitud balanceada por fuente, sexo, grupo de edad, localización y `dx_type`, con IC por bootstrap
sobre lesiones y diferencia frente al resto (familia exploratoria con Benjamini-Hochberg).
`pocas_imagenes` marca las categorías con menos de `evaluation.subgroup_min_n` imágenes.

In [ ]:
benchmark.cached(cfg, definitivas, 'subgrupos', lambda: benchmark.subgroup_family(definitivas, cfg), RECALCULAR)

### 12.5 Diagnóstico de atajos

Exactitud balanceada con y sin cada artefacto de `data/anotacion_artefactos.csv` en las imágenes anotadas
de cada partición de `errors.shortcut_splits`; en validación se usan las predicciones fuera de pliegue.

In [ ]:
show({s: errors.shortcut_table(definitivas, cfg, profundos, s) for s in cfg["errors"]["shortcut_splits"]})

### 12.6 Taxonomía de errores

Muestra estratificada por clase real de `errors.taxonomy_n` errores del mejor modelo en
`data/taxonomia_errores.csv`, con una columna 0/1 por categoría de `errors.taxonomy_categories` para el
etiquetado manual, y resumen de las etiquetas.

In [ ]:
plantilla = errors.taxonomy_template(cfg, datos_test, ranking, ESCRIBIR_PLANTILLA_TAXONOMIA)
show({"Categorías": pd.Series(cfg["errors"]["taxonomy_categories"], name="categoría").to_frame(),
      "Plantilla": plantilla})

In [ ]:
show(errors.taxonomy_status(cfg))
errors.taxonomy_by_class(cfg)

## 13. Análisis exploratorio: fuente de adquisición (RQ3)

Diferencia de exactitud balanceada entre `e2_stats.source_target` y el resto de fuentes por modelo, con
IC y p por bootstrap sobre lesiones y Benjamini-Hochberg, y la diferencia entre el transformador y la
media de las CNN.

In [ ]:
benchmark.cached(cfg, definitivas, 'rq3', lambda: benchmark.source_gap(definitivas, cfg), RECALCULAR)

## 14. Techo de desempeño

Exactitud balanceada de k vecinos sobre los embeddings de MobileNetV2 (sin vecinos de la misma lesión),
cota de Cover y Hart del error de Bayes y cota de lectores, y γ = (s_techo − s_mejor) / (s_techo − s_ref).

In [ ]:
knn = ceiling.knn_ceiling(cfg)
show({"k vecinos": knn, "γ": benchmark.gamma_table(definitivas, cfg, ceiling.ceilings_table(cfg, knn))})

## 15. Costo y frontera de Pareto

Costo medido con datos sintéticos (`python -m src.cost --all`) y tiempos de ajuste y entrenamiento de las
corridas; frontera de Pareto de la métrica primaria frente a cada costo y eficiencia relativa
η = (s_k / C_k) / (s_base / C_base) con IC por bootstrap sobre lesiones.

In [ ]:
costo

In [ ]:
pareto = benchmark.pareto_table(definitivas, cfg, costo)
show(benchmark.plot_pareto(pareto, cfg))
show({"Frontera de Pareto": pareto, "Eficiencia relativa": benchmark.efficiency_table(definitivas, cfg, costo)})

## 16. Réplica de pinar2026

ResNet-50 frente a la exactitud balanceada publicada (0,686 [0,653; 0,721]) con el criterio del
[capítulo 14](14-paper-guia.md): exitosa si el valor cae dentro del IC 95 % por t de la media de las
corridas definitivas o si la diferencia es menor que δ.

In [ ]:
benchmark.replication_table(definitivas, cfg)

## 17. Limitación del paper guía

Diferencia de exactitud balanceada entre vidir_molemax y el resto de fuentes en los tres modelos base,
con IC 95 % por bootstrap sobre lesiones.

In [ ]:
benchmark.source_gap(definitivas, cfg, grupos["base_models"])